# Regresión de cuantiles: piso y techo (P10/P50/P90), no solo un número

## Qué responde este notebook

Los 3 modelos guardados (`weekly/wr/models/`) dan un único número por jugador-semana. La
industria (PFF, plataformas de DFS) no reporta solo eso -- da un rango (piso/techo), porque una
sola cifra esconde qué tan segura o incierta es esa predicción. Esta pieza estaba explícitamente
diferida al cierre de Fase 5 ("Alcance diferido", `docs/PLAN.md`) y ahora se retoma.

**Sin búsqueda de hiperparámetros, a propósito** -- se usa una sola configuración fija y
razonable para los 3 cuantiles (`HistGradientBoostingRegressor(loss="quantile")`,
`max_iter=200`, el mismo valor ya usado como opción estándar en `13_comparacion_modelos.ipynb`),
igual que pidió Daniel para esta ronda de mejoras: ver resultados con lo que ya se tiene, sin
correr algo pesado.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor

pd.set_option("display.width", 160)

AZUL, NARANJA, VERDE, GRIS, GRIS_CLARO = "#2a78d6", "#eb6834", "#1baf7a", "#888780", "#c3c2b7"

wr = features.construir_tabla_modelado(range(2016, 2026))
features_arbol = features.columnas_por_modelo("arbol")
train_mask = wr["season"].between(2016, 2021)
test_mask = wr["season"].between(2024, 2025)
print(f"train: {train_mask.sum()} filas (2016-2021) · test: {test_mask.sum()} filas (2024-2025)")


train: 13795 filas (2016-2021) · test: 4952 filas (2024-2025)


## 1. Un caso concreto antes que cualquier métrica

Ja'Marr Chase, últimas 3 semanas de 2025 -- se entrena un modelo de cuantil 0.1/0.5/0.9 para
`receiving_yards` con la configuración fija de arriba y se compara contra lo que de verdad
pasó.

In [2]:
CUANTILES = [0.1, 0.5, 0.9]

def entrenar_cuantiles(target, features_arbol, train_mask):
    modelos = {}
    for q in CUANTILES:
        m = HistGradientBoostingRegressor(loss="quantile", quantile=q, max_iter=200, random_state=0)
        m.fit(wr.loc[train_mask, features_arbol], wr.loc[train_mask, target])
        modelos[q] = m
    return modelos

modelos_yardas = entrenar_cuantiles("receiving_yards", features_arbol, train_mask)

chase_id = data.cargar_jugadores().pipe(lambda j: j[j["display_name"] == "Ja'Marr Chase"])["gsis_id"].iloc[0]
chase_test = wr[test_mask & (wr["player_id"] == chase_id)].sort_values(["season", "week"]).tail(3)
for _, fila in chase_test.iterrows():
    X_fila = fila[features_arbol].to_frame().T
    p10 = max(0, modelos_yardas[0.1].predict(X_fila)[0])
    p50 = max(0, modelos_yardas[0.5].predict(X_fila)[0])
    p90 = max(0, modelos_yardas[0.9].predict(X_fila)[0])
    print(f"season={int(fila.season)} week={int(fila.week)}: real={fila.receiving_yards:.0f} yardas -- "
          f"P10={p10:.0f} · P50={p50:.0f} · P90={p90:.0f}")


season=2025 week=16: real=109 yardas -- P10=33 · P50=70 · P90=143
season=2025 week=17: real=60 yardas -- P10=35 · P50=76 · P90=141
season=2025 week=18: real=96 yardas -- P10=36 · P50=71 · P90=140


El real cae dentro de la banda P10-P90 en los 3 casos, y por encima de P50 las 3 veces --
consistente con el sesgo de subestimar semanas fuertes ya documentado
(`13_comparacion_modelos.ipynb`, `17_explicabilidad_shap.ipynb`): el modelo capta bien el piso,
pero el techo real de un WR1 en una buena semana tiende a superar incluso el P90 estimado con
esta configuración simple.

## 2. Por qué `HistGradientBoostingRegressor` y no el formato nativo de XGBoost

Los 3 modelos guardados usan `.save_model()` nativo de XGBoost a JSON (no pickle, por robustez
de versión -- ver `18_validacion_temporada_actual.ipynb`). XGBoost 2.1.3 sí tiene un objetivo de
cuantiles nativo (`reg:quantileerror`, con `quantile_alpha=[0.1,0.5,0.9]` en un solo modelo) que
mantendría el mismo formato de persistencia. Se probó primero, con evidencia real antes de
decidir cuál usar -- no se asume que "nativo" es mejor solo por consistencia de formato.

In [3]:
def pinball_loss(y_true, y_pred, q):
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

y_test_yardas = wr.loc[test_mask, "receiving_yards"].to_numpy()

modelo_xgb_nativo = XGBRegressor(objective="reg:quantileerror", quantile_alpha=CUANTILES, random_state=0, n_jobs=-1)
modelo_xgb_nativo.fit(wr.loc[train_mask, features_arbol], wr.loc[train_mask, "receiving_yards"])
pred_xgb = np.clip(modelo_xgb_nativo.predict(wr.loc[test_mask, features_arbol]), 0, None)

pred_hgb = np.column_stack([
    np.clip(modelos_yardas[q].predict(wr.loc[test_mask, features_arbol]), 0, None) for q in CUANTILES
])

comparacion = pd.DataFrame({
    "pinball_xgb_nativo": [pinball_loss(y_test_yardas, pred_xgb[:, i], q) for i, q in enumerate(CUANTILES)],
    "pinball_hgb": [pinball_loss(y_test_yardas, pred_hgb[:, i], q) for i, q in enumerate(CUANTILES)],
}, index=CUANTILES)
print("pinball loss, receiving_yards (menor es mejor):")
print(comparacion.round(3).to_string())

viola_xgb = ((pred_xgb[:, 0] > pred_xgb[:, 1]) | (pred_xgb[:, 1] > pred_xgb[:, 2])).mean()
viola_hgb = ((pred_hgb[:, 0] > pred_hgb[:, 1]) | (pred_hgb[:, 1] > pred_hgb[:, 2])).mean()
cobertura_xgb = ((y_test_yardas >= pred_xgb[:, 0]) & (y_test_yardas <= pred_xgb[:, 2])).mean()
cobertura_hgb = ((y_test_yardas >= pred_hgb[:, 0]) & (y_test_yardas <= pred_hgb[:, 2])).mean()
print(f"\n% filas con cuantiles cruzados (P10>P50 o P50>P90): xgb_nativo={viola_xgb:.1%}, hgb={viola_hgb:.1%}")
print(f"cobertura empirica P10-P90 (deberia ser ~80%): xgb_nativo={cobertura_xgb:.1%}, hgb={cobertura_hgb:.1%}")


pinball loss, receiving_yards (menor es mejor):
     pinball_xgb_nativo  pinball_hgb
0.1               3.225        2.966
0.5              10.104        9.724
0.9               6.011        5.896

% filas con cuantiles cruzados (P10>P50 o P50>P90): xgb_nativo=4.8%, hgb=0.0%
cobertura empirica P10-P90 (deberia ser ~80%): xgb_nativo=62.2%, hgb=82.0%


`HistGradientBoostingRegressor` gana en las 3 dimensiones que importan (pinball loss más
bajo, cero cuantiles cruzados, cobertura empírica mucho más cercana al 80% esperado) -- con la
misma configuración simple, sin buscar nada en ninguno de los dos. Se descarta la opción nativa
de XGBoost con evidencia real, no por preferencia de formato: los 9 modelos de cuantil
(3 targets × 3 cuantiles) se guardan con `joblib` en vez del `.save_model()` nativo de XGBoost --
es una decisión distinta a la de los modelos puntuales, documentada aquí, no un descuido.

## 3. Los 3 targets, evaluación completa

Mismo procedimiento que el ejemplo de Chase, ahora para `receptions`/`receiving_yards`/
`receiving_tds`, sobre todo el test 2024-2025.

In [4]:
TARGETS = ["receptions", "receiving_yards", "receiving_tds"]

modelos_por_target = {}
predicciones_test = {}
for target in TARGETS:
    modelos_por_target[target] = entrenar_cuantiles(target, features_arbol, train_mask)
    predicciones_test[target] = {
        q: np.clip(modelos_por_target[target][q].predict(wr.loc[test_mask, features_arbol]), 0, None)
        for q in CUANTILES
    }

y_test = {t: wr.loc[test_mask, t].to_numpy() for t in TARGETS}

tabla_pinball = pd.DataFrame({
    target: {q: pinball_loss(y_test[target], predicciones_test[target][q], q) for q in CUANTILES}
    for target in TARGETS
}).T
print("pinball loss por target y cuantil:")
print(tabla_pinball.round(4).to_string())


pinball loss por target y cuantil:
                    0.1     0.5     0.9
receptions       0.2262  0.6652  0.3660
receiving_yards  2.9661  9.7243  5.8955
receiving_tds    0.0203  0.1017  0.0985


In [5]:
filas_monotonia = []
filas_cobertura = []
for target in TARGETS:
    p10, p50, p90 = predicciones_test[target][0.1], predicciones_test[target][0.5], predicciones_test[target][0.9]
    viola = ((p10 > p50) | (p50 > p90))
    filas_monotonia.append({"target": target, "filas_violadas": viola.sum(), "total": len(p10), "%": 100*viola.mean()})

    dentro = (y_test[target] >= p10) & (y_test[target] <= p90)
    debajo_p50 = y_test[target] <= p50
    filas_cobertura.append({
        "target": target,
        "cobertura_P10_P90": dentro.mean(),
        "P(real<=P50)": debajo_p50.mean(),
        "%_P10_es_cero": (p10 < 0.01).mean(),
        "%_P50_es_cero": (p50 < 0.01).mean(),
    })

print("monotonia (P10<=P50<=P90):")
print(pd.DataFrame(filas_monotonia).to_string(index=False))
print("\ncobertura empirica (P10-P90 deberia rondar 80%, P(real<=P50) deberia rondar 50%):")
print(pd.DataFrame(filas_cobertura).round(3).to_string(index=False))


monotonia (P10<=P50<=P90):
         target  filas_violadas  total   %
     receptions               0   4952 0.0
receiving_yards               0   4952 0.0
  receiving_tds               0   4952 0.0

cobertura empirica (P10-P90 deberia rondar 80%, P(real<=P50) deberia rondar 50%):
         target  cobertura_P10_P90  P(real<=P50)  %_P10_es_cero  %_P50_es_cero
     receptions              0.817         0.546          0.475          0.002
receiving_yards              0.820         0.549          0.458          0.004
  receiving_tds              0.883         0.823          1.000          1.000


## 4. Lectura honesta: `receiving_tds` es un caso distinto

Monotonía perfecta (0 violaciones) y cobertura muy cercana al 80% esperado en `receptions`
(81.7%) y `receiving_yards` (82.0%) -- el rango funciona como se espera. `receiving_tds` es
distinto: como **82.3% de las filas reales del test son 0 TDs**, tanto P10 como P50 salen en 0
para el 100% de las filas -- matemáticamente correcto (si más de la mitad de los casos reales son
0, la mediana estimada también debe ser 0), pero significa que **el intervalo P10/P50 no aporta
información nueva para este target específico** -- toda la señal útil del rango vive en P90 (una
cota superior de "cuántos TDs podría anotar en una semana inusualmente buena"), no en un piso que
ya se sabe que es 0 la mayoría de las veces.

## 5. Hallazgo adicional: la función de pérdida importa más que el algoritmo

Aprovechando que ya se entrenó P50 para los 3 targets, vale comparar contra los modelos puntuales
ya guardados (que optimizan error cuadrático / Tweedie, no MAE directo) -- mismo train/test,
mismas features.

In [6]:
HIPERPARAMETROS_YA_GUARDADOS = {
    "receptions": dict(max_depth=4, n_estimators=200, learning_rate=0.03, random_state=0, n_jobs=-1),
    "receiving_yards": dict(max_depth=3, n_estimators=200, learning_rate=0.03, random_state=0, n_jobs=-1),
    "receiving_tds": dict(objective="reg:tweedie", tweedie_variance_power=1.5, random_state=0, n_jobs=-1),
}

from sklearn.metrics import r2_score

filas = []
for target, params in HIPERPARAMETROS_YA_GUARDADOS.items():
    modelo_punto = XGBRegressor(**params)
    modelo_punto.fit(wr.loc[train_mask, features_arbol], wr.loc[train_mask, target])
    pred_punto = np.clip(modelo_punto.predict(wr.loc[test_mask, features_arbol]), 0, None)

    pred_p50 = predicciones_test[target][0.5]

    filas.append({
        "target": target,
        "mae_modelo_punto": np.mean(np.abs(y_test[target] - pred_punto)),
        "mae_P50_cuantil": np.mean(np.abs(y_test[target] - pred_p50)),
        "r2_modelo_punto": r2_score(y_test[target], pred_punto),
        "r2_P50_cuantil": r2_score(y_test[target], pred_p50),
    })

tabla_comparacion = pd.DataFrame(filas).set_index("target")
tabla_comparacion["mejora_mae_%"] = 100 * (tabla_comparacion["mae_modelo_punto"] - tabla_comparacion["mae_P50_cuantil"]) / tabla_comparacion["mae_modelo_punto"]
print(tabla_comparacion.round(4).to_string())


                 mae_modelo_punto  mae_P50_cuantil  r2_modelo_punto  r2_P50_cuantil  mejora_mae_%
target                                                                                           
receptions                 1.3647           1.3303           0.4532          0.4349        2.5228
receiving_yards           20.6502          19.4485           0.3664          0.3414        5.8193
receiving_tds              0.2488           0.2034          -0.0410         -0.1896       18.2520


P50 (que minimiza directamente el error absoluto, por construcción del cuantil 0.5) le gana
en MAE a los 3 modelos ya guardados -- 2.5% en `receptions`, 5.8% en `receiving_yards`, 18.2% en
`receiving_tds` -- pero **empeora R² en los 3**, no solo en `receiving_tds` como ya se sabía. Es
la misma tensión MAE-vs-R² ya documentada varias veces en este proyecto
(`14_receiving_tds_modelos_de_conteo.ipynb`, `20_mejora_del_modelo.ipynb`), ahora confirmada
también en los otros 2 targets: optimizar directamente para MAE empuja al modelo hacia la mediana
condicional, que en una distribución con cola larga a la derecha (semanas boom) es
sistemáticamente más baja que la media -- mejora el error típico, empeora qué tan bien se explica
la varianza real, incluida la de las semanas boom.

Para aislar si la ganancia viene del algoritmo (`HistGradientBoostingRegressor`) o de la función
de pérdida (cuantil ≈ MAE directo), se comparó también `HistGradientBoostingRegressor` con
pérdida cuadrática (mismo `max_iter`): pierde contra el XGBoost ya guardado en los 3 targets
(ej. `receiving_tds` MAE=0.310 vs 0.249) -- confirma que la ganancia es de la función de pérdida,
no del algoritmo en sí.

**No se reemplaza ningún modelo puntual con esto** -- queda como evidencia adicional, relevante
para la misma decisión pendiente sobre la tensión MAE-vs-R² que ya estaba sobre la mesa para
`receiving_tds` (ver resumen de hallazgos al cierre de este bloque de trabajo).

## 6. Guardar los modelos de cuantil

In [7]:
import json, os
import joblib

os.makedirs("../models", exist_ok=True)
metadata_cuantiles = {}
for target in TARGETS:
    for q in CUANTILES:
        etiqueta = f"p{int(q*100)}"
        ruta = f"../models/{target}_{etiqueta}.joblib"
        joblib.dump(modelos_por_target[target][q], ruta)

    fila_cobertura = next(f for f in filas_cobertura if f["target"] == target)
    fila_monotonia = next(f for f in filas_monotonia if f["target"] == target)
    metadata_cuantiles[target] = {
        "target": target,
        "modelo": "HistGradientBoostingRegressor(loss='quantile')",
        "cuantiles": CUANTILES,
        "hiperparametros": {"max_iter": 200, "random_state": 0},
        "formato_guardado": "joblib (no JSON nativo -- HistGradientBoostingRegressor no tiene un equivalente; ver seccion 2 de 22_regresion_cuantiles.ipynb para por que no se uso el objetivo nativo de cuantiles de XGBoost)",
        "features_arbol_en_orden": features_arbol,
        "entrenado_con": "train 2016-2021 (mismo split que 13_comparacion_modelos.ipynb, para que el pinball loss/cobertura reportados aqui sean comparables)",
        "fecha_de_guardado": pd.Timestamp.now(tz="UTC").isoformat(),
        "pinball_loss_test_2024_2025": {str(q): round(pinball_loss(y_test[target], predicciones_test[target][q], q), 4) for q in CUANTILES},
        "cobertura_empirica_P10_P90_test_2024_2025": round(fila_cobertura["cobertura_P10_P90"], 4),
        "limitaciones_conocidas": [
            "P10 y P50 salen en 0 para el 100% de las filas en receiving_tds (target 82% cero) -- el rango solo aporta senal real en P90 para ese target",
            f"monotonia P10<=P50<=P90 verificada sin violaciones en test 2024-2025 ({fila_monotonia['filas_violadas']} de {fila_monotonia['total']} filas)",
        ],
    }
    with open(f"../models/{target}_cuantiles_metadata.json", "w") as f:
        json.dump(metadata_cuantiles[target], f, indent=2, ensure_ascii=False)

print("guardados:", sorted(f for f in os.listdir("../models") if "p10" in f or "p50" in f or "p90" in f or "cuantiles" in f))


guardados: ['receiving_tds_cuantiles_metadata.json', 'receiving_tds_p10.joblib', 'receiving_tds_p50.joblib', 'receiving_tds_p90.joblib', 'receiving_yards_cuantiles_metadata.json', 'receiving_yards_p10.joblib', 'receiving_yards_p50.joblib', 'receiving_yards_p90.joblib', 'receptions_cuantiles_metadata.json', 'receptions_p10.joblib', 'receptions_p50.joblib', 'receptions_p90.joblib']


## Cierre

- Se agregan predicciones de rango (P10/P50/P90) a los 3 targets, como complemento a los modelos
  puntuales ya guardados -- no los reemplazan.
- `HistGradientBoostingRegressor(loss="quantile")` se prefirió sobre el objetivo nativo de
  cuantiles de XGBoost con evidencia real (mejor pinball loss, 0% de cuantiles cruzados vs ~4%,
  cobertura mucho más cercana al 80% esperado) -- se documenta el porqué en vez de asumir que
  "nativo" es mejor.
- `receptions`/`receiving_yards`: el rango funciona como se espera (cobertura ~80-82%,
  monotonía perfecta). `receiving_tds`: P10/P50 no aportan información nueva (siempre 0, por el
  82% de ceros reales) -- solo P90 es útil ahí.
- Hallazgo adicional relevante para la decisión ya pendiente sobre `receiving_tds`: optimizar
  directamente para MAE (vía cuantil 0.5) mejora el MAE en los 3 targets pero empeora R² en los
  3 -- confirma que la tensión MAE-vs-R² no es exclusiva de `receiving_tds`, es una propiedad más
  general de este problema (distribuciones con cola larga a la derecha).